# Stage 6: final synthesis and robustness review

This notebook combines only saved Stage 1–5 results. It does not run transport simulations. Existing source functions verify saved Stage 4 channel counts and the Stage 5 QPC potential.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np

project_root = Path.cwd()
if not (project_root / "results").is_dir():
    project_root = project_root.parent
assert (project_root / "results").is_dir()
sys.path.insert(0, str(project_root / "src"))
from quantum_transport import open_mode_count, transverse_modes, quantum_point_contact_potential
from quantum_transport.synthesis import analyze_saved_results, create_overview_figure, load_csv, write_final_summary

results_dir = project_root / "results"
stage_files = {
    "stage1": ["stage1_transmission.csv"],
    "stage2": ["stage2_transmission.csv", "stage2_dos.csv", "stage2_resonances.csv", "stage2_resonance_ldos.csv"],
    "stage3": ["stage3_disorder_realizations.csv", "stage3_single_realization.csv", "stage3_ensemble.csv", "stage3_length_scaling.csv"],
    "stage4": ["stage4_modes.csv", "stage4_multichannel.csv"],
    "stage5": ["stage5_qpc_gate_scan.csv", "stage5_qpc_ldos.csv", "stage5_qpc_potential.csv", "stage5_qpc_transmission.csv"],
}
saved_tables = {stage: {name: load_csv(results_dir / name) for name in names}
                for stage, names in stage_files.items()}
metadata = {stage: json.loads((results_dir / f"{stage}_metadata.json").read_text(encoding="utf-8"))
            for stage in ("stage1", "stage2", "stage3", "stage4")}
metadata["stage5"] = json.loads((results_dir / "stage5_qpc_metadata.json").read_text(encoding="utf-8"))
{stage: {name: len(rows) for name, rows in tables.items()} for stage, tables in saved_tables.items()}

In [ ]:
# Recompute the clean Stage 4 analytic open-channel count using the existing module.
stage4_rows = saved_tables["stage4"]["stage4_multichannel.csv"]
mode_energies, _ = transverse_modes(metadata["stage4"]["geometry"]["transverse_width_Ly"])
energies4 = np.asarray([float(row["energy"]) for row in stage4_rows])
computed_counts = open_mode_count(energies4, mode_energies, hopping=1.0)
saved_counts = np.asarray([int(row["N_open"]) for row in stage4_rows])
np.testing.assert_array_equal(computed_counts, saved_counts)

# Rebuild the Stage 5 gate potential with its source module and compare the saved map.
qpc_meta = metadata["stage5"]
gate = 1.0
potential = qpc_meta["qpc_potential"]
rebuilt = quantum_point_contact_potential(
    qpc_meta["geometry"]["Lx"], qpc_meta["geometry"]["Ly"], gate,
    longitudinal_sigma=potential["longitudinal_sigma_sites"],
    transverse_strength=potential["transverse_strength_alpha"],
)
potential_rows = saved_tables["stage5"]["stage5_qpc_potential.csv"]
saved_potential = np.empty_like(rebuilt)
for row in potential_rows:
    saved_potential[int(row["x"]), int(row["y"])] = float(row["onsite_energy"])
np.testing.assert_allclose(rebuilt, saved_potential, rtol=1e-12, atol=1e-12)
print(f"Stage 4 mode counts match at {len(stage4_rows)} saved energies.")
print(f"Stage 5 gate potential matches the saved {rebuilt.size}-site map.")

In [ ]:
analysis = analyze_saved_results(results_dir)
analysis["metrics"]

## Interpretation

Stage 1 checks the matched-chain reference across safe interior windows. Stage 2 compares the saved resonance with two detuned LDOS samples. Stage 3 reports finite-ensemble statistics and length trends without thermodynamic extrapolation. Stage 4 uses the saved 0.05 threshold exclusion. Stage 5 gate scans and transmission bounds are checked against incoming lead modes; its LDOS is spectral density, not current density. All claims are limited to the saved grids, geometries, and samples.

In [ ]:
# Recreate the summary and overview figure from existing data.
analysis = write_final_summary(results_dir)
create_overview_figure(results_dir, project_root / "figures" / "final_project_overview.png")
[(row["stage"], row["principal_numerical_result"]) for row in analysis["summary_rows"]]